# The MUSIC Algorithm
## Tutorial 7: Multiple Signal Classification

MUSIC (Schmidt, 1986) is the prototype high-resolution DOA estimator.  It exploits **eigendecomposition** to separate signal and noise subspaces, achieving super-resolution that far exceeds classical beamforming.

Topics:
1. **MUSIC principle** – noise-subspace orthogonality
2. **MUSIC spectrum computation**
3. **Source number detection** – AIC and MDL
4. **Performance at different SNR and N**
5. **Forward-backward averaging**

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from doa_methods.array_processing import UniformLinearArray, SignalModel
from doa_methods.subspace import MUSIC
from doa_methods.classical import ConventionalBeamforming, CaponBeamforming

plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 12

M = 16
array = UniformLinearArray(M=M, d=0.5)
sm    = SignalModel(array)
music = MUSIC(array)
cbf   = ConventionalBeamforming(array)
capon = CaponBeamforming(array)

angle_grid = np.linspace(-np.pi/2, np.pi/2, 3601)

print("Setup complete.")

## 1. MUSIC Principle

**Key idea**: the noise eigenvectors span a subspace orthogonal to every true steering vector.

$$P_{\text{MUSIC}}(\theta) = \frac{1}{\mathbf{a}^H(\theta)\mathbf{U}_n\mathbf{U}_n^H\mathbf{a}(\theta)}$$

- For $\theta = \theta_k$ (a true DOA): denominator $\to 0$ → spectrum $\to \infty$ → **sharp peak**
- For other $\theta$: denominator > 0 → spectrum remains low

This is the noise-subspace **pseudo-spectrum** (not a true power spectrum).

In [ ]:
doas_true = np.deg2rad([-20.0, 15.0])
K = len(doas_true)
snr_db = 10
N = 200

X, _, _ = sm.generate_signals(doas_true, N, snr_db, seed=42)
R = X @ X.conj().T / N

# Eigendecomposition
eigenvals, eigenvecs = np.linalg.eigh(R)
eigenvals = eigenvals[::-1]; eigenvecs = eigenvecs[:, ::-1]
U_n = eigenvecs[:, K:]   # noise subspace

# MUSIC spectrum (vectorised)
A = array.array_manifold(angle_grid)                       # M × Ngrid
proj = np.sum(np.abs(A.conj().T @ U_n)**2, axis=1)        # Ngrid
music_spec = 1.0 / (proj + 1e-12)
music_spec_dB = 10*np.log10(music_spec / music_spec.max() + 1e-12)

# For comparison: CBF and Capon
bp_cbf   = cbf.beam_pattern(X, angle_grid)
bp_cbf_dB = 10*np.log10(bp_cbf / bp_cbf.max() + 1e-12)
bp_capon = capon.beam_pattern(X, angle_grid)
bp_capon_dB = 10*np.log10(bp_capon / bp_capon.max() + 1e-12)

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(np.rad2deg(angle_grid), bp_cbf_dB,   'b-',  lw=1.5, alpha=0.8, label='CBF')
ax.plot(np.rad2deg(angle_grid), bp_capon_dB, 'g-',  lw=1.5, alpha=0.8, label='Capon')
ax.plot(np.rad2deg(angle_grid), music_spec_dB, 'r-', lw=2.5,             label='MUSIC')
for th in doas_true:
    ax.axvline(np.rad2deg(th), color='k', ls='--', alpha=0.5)
ax.set_xlabel('θ (°)'); ax.set_ylabel('Normalised Spectrum (dB)')
ax.set_title(f'CBF vs Capon vs MUSIC  (M={M}, SNR={snr_db} dB, N={N})')
ax.set_ylim(-40, 2); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

doas_music = music.estimate(X, K=K, angle_grid=angle_grid)
print(f"True DOAs : {np.round(np.rad2deg(doas_true), 3)} °")
print(f"MUSIC est : {np.round(np.rad2deg(doas_music), 3)} °")

## 2. Effect of Knowing K vs Estimating It

MUSIC requires $K$.  In practice $K$ is unknown and must be estimated from the data.

### Information-Theoretic Criteria

**AIC** (Akaike):
$$\text{AIC}(k) = -2N\sum_{i=k+1}^{M}\ln\hat{\lambda}_i + 2k(2M-k)$$

**MDL** (Minimum Description Length / BIC):
$$\text{MDL}(k) = -N\sum_{i=k+1}^{M}\ln\hat{\lambda}_i + \frac{1}{2}k(2M-k)\ln N$$

The minimiser of each criterion is the estimated $K$.

In [ ]:
from doa_methods.utils.math_utils import source_number_estimation

N_aic = 200
X_aic, _, _ = sm.generate_signals(doas_true, N_aic, snr_db=10, seed=0)

K_aic = music.source_number_estimation(X_aic, method='aic')
K_mdl = music.source_number_estimation(X_aic, method='mdl')
K_gap = music.source_number_estimation(X_aic, method='eigenvalue_gap')

print(f"True K    = {K}")
print(f"AIC  K̂   = {K_aic}")
print(f"MDL  K̂   = {K_mdl}")
print(f"Gap  K̂   = {K_gap}")

# Visualise eigenvalue spectrum with detected K
R_aic = X_aic @ X_aic.conj().T / N_aic
evals, _ = np.linalg.eigh(R_aic)
evals = evals[::-1]

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(range(1, M+1), evals, 'ko-', ms=7, lw=2)
ax.axvline(K     + 0.5, color='r',      ls='--', label=f'True K={K}')
ax.axvline(K_aic + 0.5, color='orange', ls=':',  label=f'AIC  K̂={K_aic}')
ax.axvline(K_mdl + 0.5, color='green',  ls='-.',  label=f'MDL  K̂={K_mdl}')
ax.set_xlabel('Index'); ax.set_ylabel('Eigenvalue')
ax.set_title('Source Number Estimation from Eigenvalue Spectrum')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. MUSIC vs SNR

At very low SNR the noise and signal eigenvalues **overlap** and the subspace estimate degrades — MUSIC breaks down below a threshold SNR.

In [ ]:
snr_range = np.arange(-10, 26, 3)
n_trials  = 200
N_fixed   = 200

def rmse_music(doas, N, snr, trials):
    K_ = len(doas)
    errs = []
    for t in range(trials):
        X_, _, _ = sm.generate_signals(doas, N, snr, seed=t)
        try:
            est = np.sort(music.estimate(X_, K=K_, angle_grid=angle_grid))
            errs.append(np.sqrt(np.mean((est - np.sort(doas))**2)))
        except Exception:
            errs.append(np.pi)
    return np.rad2deg(np.sqrt(np.mean(np.array(errs)**2)))

rmse_mu = [rmse_music(doas_true, N_fixed, s, n_trials) for s in snr_range]

from doa_methods.classical import ConventionalBeamforming as CBF_cls
cbf_inst = CBF_cls(array)

def rmse_cbf_fn(doas, N, snr, trials):
    K_ = len(doas)
    errs = []
    for t in range(trials):
        X_, _, _ = sm.generate_signals(doas, N, snr, seed=t)
        try:
            est = np.sort(cbf_inst.estimate(X_, K=K_))
            errs.append(np.sqrt(np.mean((est - np.sort(doas))**2)))
        except Exception:
            errs.append(np.pi)
    return np.rad2deg(np.sqrt(np.mean(np.array(errs)**2)))

rmse_c = [rmse_cbf_fn(doas_true, N_fixed, s, n_trials) for s in snr_range]

fig, ax = plt.subplots(figsize=(11, 6))
ax.semilogy(snr_range, rmse_c,  'b-o', ms=6, lw=2, label='CBF')
ax.semilogy(snr_range, rmse_mu, 'r-s', ms=6, lw=2, label='MUSIC')
ax.set_xlabel('SNR (dB)'); ax.set_ylabel('RMSE (°)')
ax.set_title(f'RMSE vs SNR: CBF vs MUSIC  (M={M}, N={N_fixed})')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Forward-Backward Averaging

For **real-valued** arrays and **uncorrelated** sources, the covariance matrix has a special centro-Hermitian structure.  Forward-backward (FB) averaging exploits this to effectively **double the snapshot count**:

$$\hat{\mathbf{R}}_{FB} = \frac{1}{2}\left(\hat{\mathbf{R}} + \mathbf{J}\hat{\mathbf{R}}^*\mathbf{J}\right)$$

where $\mathbf{J}$ is the exchange matrix.  This also helps with **slightly correlated** sources.

In [ ]:
N_small = 30    # very few snapshots
X_fb, _, _ = sm.generate_signals(doas_true, N_small, snr_db=10, seed=77)

spec_no_fb = music.estimate(X_fb, K=K, angle_grid=angle_grid)
spec_fb    = music.estimate(X_fb, K=K, angle_grid=angle_grid, use_fb_averaging=True)

# Compute both spectra for plotting
def get_music_spec(X, K, use_fb):
    R_ = X @ X.conj().T / X.shape[1]
    if use_fb:
        from doa_methods.utils.math_utils import forward_backward_averaging
        R_ = forward_backward_averaging(R_)
    evals_, evecs_ = np.linalg.eigh(R_)
    U_n_ = evecs_[:, ::-1][:, K:]
    A_ = array.array_manifold(angle_grid)
    p = np.sum(np.abs(A_.conj().T @ U_n_)**2, axis=1)
    spec = 1/(p+1e-12)
    return 10*np.log10(spec/spec.max()+1e-12)

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(np.rad2deg(angle_grid), get_music_spec(X_fb, K, False),
        'b-', lw=2, label='MUSIC (no FB)')
ax.plot(np.rad2deg(angle_grid), get_music_spec(X_fb, K, True),
        'r-', lw=2, label='MUSIC (with FB)')
for th in doas_true:
    ax.axvline(np.rad2deg(th), color='k', ls='--', alpha=0.6)
ax.set_xlabel('θ (°)'); ax.set_ylabel('Normalised Spectrum (dB)')
ax.set_title(f'FB Averaging Effect  (N={N_small}, SNR=10 dB)')
ax.set_ylim(-40, 2); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Summary

- MUSIC provides **super-resolution** well beyond the Rayleigh limit.
- Requires knowledge of (or an estimate of) the source number $K$.
- AIC and MDL are standard estimators; MDL tends to be more conservative.
- FB averaging improves performance when snapshots are limited.
- MUSIC breaks down below a threshold SNR when signal and noise eigenvalues merge.

## Exercises
1. Show that the MUSIC pseudo-spectrum is **not** a true power spectral density (hint: what are the units?).
2. Fix $K=2$, $\text{SNR}=15$ dB, $N=100$, and vary the angular separation from $1°$ to $15°$ in steps of $1°$.  Find the minimum separation where MUSIC still resolves both sources (RMSE < 1°).
3. Prove that FB averaging preserves the signal subspace for a ULA with symmetric steering vectors.